<a href="https://colab.research.google.com/github/snig-17/COMP0014/blob/main/COMP0014_Final_Coursework_Submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup and Data Loading

In [1]:
import numpy as np
import struct
from array import array
import os.path
import random
import matplotlib.pyplot as plt
import time
import datetime
from sklearn import svm, metrics
from sklearn.model_selection import train_test_split
import pandas as pd
%matplotlib inline

# Fashion MNIST labels
fashion_labels = {
    0: 'T-shirt/top', 1: 'Trouser', 2: 'Pullover', 3: 'Dress', 4: 'Coat',
    5: 'Sandal', 6: 'Shirt', 7: 'Sneaker', 8: 'Bag', 9: 'Ankle boot'
}

# Data Loader Utility
class MnistDataloader(object):
    def __init__(self, training_images_filepath, training_labels_filepath,
                 test_images_filepath, test_labels_filepath):
        self.training_images_filepath = training_images_filepath
        self.training_labels_filepath = training_labels_filepath
        self.test_images_filepath = test_images_filepath
        self.test_labels_filepath = test_labels_filepath

    def read_images_labels(self, images_filepath, labels_filepath):
        with open(labels_filepath, 'rb') as file:
            magic, size = struct.unpack(">II", file.read(8))
            labels = np.array(array("B", file.read()), dtype=np.uint8)
        with open(images_filepath, 'rb') as file:
            magic, size, rows, cols = struct.unpack(">IIII", file.read(16))
            image_data = array("B", file.read())
            images = np.array(image_data, dtype=np.uint8).reshape(size, rows * cols)
        return images, labels

    def load_data(self):
        (x_train, y_train) = self.read_images_labels(self.training_images_filepath, self.training_labels_filepath)
        (x_test, y_test) = self.read_images_labels(self.test_images_filepath, self.test_labels_filepath)
        return (x_train, y_train), (x_test, y_test)

# Visualization Utility
def show_images(images, title_texts):
    cols = 5
    rows = int(len(images) / cols) + 1
    plt.figure(figsize=(20, 10))
    for i, (img, title) in enumerate(zip(images, title_texts), 1):
        plt.subplot(rows, cols, i)
        plt.imshow(img.reshape(28, 28), cmap='gray')
        plt.title(title, fontsize=10)
        plt.axis('off')
    plt.show()

# Best Result Utility
def get_best(results):
    return max(results, key=lambda x: x['test_acc'])

# Load Dataset
input_path = 'Fashion_MNIST'
loader = MnistDataloader(
    os.path.join(input_path, 'train-images-idx3-ubyte'),
    os.path.join(input_path, 'train-labels-idx1-ubyte'),
    os.path.join(input_path, 't10k-images-idx3-ubyte'),
    os.path.join(input_path, 't10k-labels-idx1-ubyte')
)
(x_train, y_train), (x_test, y_test) = loader.load_data()

## Data Preprocessing

In [2]:

# Normalize pixels
x_train = x_train / 255.0
x_test = x_test / 255.0

# Subsample for faster grid search (10% training data)
x_train_mini, _, y_train_mini, _ = train_test_split(x_train, y_train, test_size=0.9, random_state=666)
class_names = list(fashion_labels.values())

# Linear Kernel

In [ ]:


C_values = [0.1, 1, 10]
linear_results = []

for C in C_values:
    clf = svm.SVC(kernel='linear', C=C)
    start = time.time()
    clf.fit(x_train_mini, y_train_mini)
    train_time = time.time() - start
    linear_results.append({
        'C': C, 'train_acc': clf.score(x_train_mini, y_train_mini),
        'test_acc': clf.score(x_test, y_test), 'time': train_time
    })

best_linear = get_best(linear_results)
print(f"Best Linear: {best_linear}")

In [ ]:
# Helper function to find the best model configuration based on test accuracy
def get_best(results):
    # The best model is considered the one with the highest test accuracy
    return max(results, key=lambda x: x['test_acc'])

# Get the best configuration found for the Linear kernel
best_linear  = get_best(linear_results)
#best_rbf     = get_best(rbf_results)
#best_poly    = get_best(poly_results)
#best_sigmoid = get_best(sigmoid_results)

# Print the best Linear kernel configuration
print('Best Linear: ',  best_linear)
#print('Best RBF: ',     best_rbf)
#print('Best Poly: ',    best_poly)
#print('Best Sigmoid: ', best_sigmoid)

In [ ]:
# Initialize the SVM classifier with the best hyperparameters found for the Linear kernel
classifier = svm.SVC(
    kernel='linear',
    C=best_linear['C'], # Use the optimal C value from the grid search
)

In [ ]:
#Metric 1: Training Time
start_time = datetime.datetime.now()
print('Started training at {}'.format(start_time))

# Train the final classifier on the mini training dataset
classifier.fit(x_train_mini, y_train_mini)

training_time = datetime.datetime.now() - start_time
print('Training complete. Elapsed time: {}'.format(training_time))


# Predictions on the full test set
start_time = datetime.datetime.now()
actual    = y_test # Actual labels from the test set
predicted = classifier.predict(x_test) # Predict labels for the test set
elapsed   = datetime.datetime.now() - start_time
print('Prediction elapsed time: {}'.format(elapsed))

#Metric 2: Confusion matrix with clothing category labels
class_names = list(fashion_labels.values()) # Get class names for display

# Compute the confusion matrix and normalize by true labels
confusion_matrix = metrics.confusion_matrix(actual, predicted, normalize="true")
cm_display = metrics.ConfusionMatrixDisplay(
    confusion_matrix,
    display_labels=class_names # Display class names on the matrix
)

fig, ax = plt.subplots(figsize=(12, 10)) # Create a figure and an axes for the plot
cm_display.plot(values_format=".2f", ax=ax, xticks_rotation=45) # Plot the confusion matrix
plt.title('Confusion Matrix: Fashion MNIST Linear Kernel') # Set plot title
plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show() # Display the plot

#Metric 3: Overall Test Accuracy
# Recalculate predictions and test accuracy (can be optimized by reusing 'predicted' from above)
predicted = classifier.predict(x_test)
test_accuracy = classifier.score(x_test, y_test)

#Metric 4: Overfitting Check
# Calculate training accuracy and the gap between training and test accuracy
train_accuracy = classifier.score(x_train_mini, y_train_mini)
overfit_gap = train_accuracy - test_accuracy

#Metric 5: Per-class F1-score (precision, recall, f1-score)
report = metrics.classification_report(
     y_test, predicted,
     target_names=class_names,
     output_dict=True)  # returns a dictionary for easier storage/analysis

# Summary of Performance Metrics
print(f'Train Accuracy:  {train_accuracy:.4f}')
print(f'Test Accuracy:   {test_accuracy:.4f}')
print(f'Overfitting Gap: {overfit_gap:.4f}')
print(f'Training Time:   {training_time}')
print(f'\nClassification Report:')
print(metrics.classification_report(y_test, predicted, target_names=class_names)) # Print detailed classification report

# Polynomial Kernel

In [ ]:


poly_results = []
for C in [0.1, 1, 10]:
    for deg in [3, 4]:
        clf = svm.SVC(kernel='poly', C=C, degree=deg)
        start = time.time()
        clf.fit(x_train_mini, y_train_mini)
        poly_results.append({
            'C': C, 'degree': deg, 'train_acc': clf.score(x_train_mini, y_train_mini),
            'test_acc': clf.score(x_test, y_test), 'time': time.time() - start
        })

best_poly = get_best(poly_results)
print(f"Best Poly: {best_poly}")

In [ ]:
best_poly = get_best(poly_results)
print(f"\nBest Polynomial: {best_poly}")

classifier_poly = svm.SVC(kernel='poly', C=best_poly['C'], degree=best_poly['degree'], gamma='scale')
classifier_poly.fit(x_train_mini, y_train_mini)
predicted_poly = classifier_poly.predict(x_test)

print(f"Test Accuracy: {classifier_poly.score(x_test, y_test):.4f}")
print("\nClassification Report:")
print(metrics.classification_report(y_test, predicted_poly, target_names=class_names))

cm_poly = metrics.confusion_matrix(y_test, predicted_poly, normalize='true')
fig, ax = plt.subplots(figsize=(10, 8))
metrics.ConfusionMatrixDisplay(cm_poly, display_labels=class_names).plot(ax=ax, xticks_rotation=45)
plt.title('Confusion Matrix: Polynomial Kernel')
plt.show()

In [ ]:
best_sigmoid = get_best(sigmoid_results)
print(f"\nBest Sigmoid: {best_sigmoid}")

classifier_sigmoid = svm.SVC(kernel='sigmoid', C=best_sigmoid['C'], coef0=best_sigmoid['coef0'], gamma='scale')
classifier_sigmoid.fit(x_train_mini, y_train_mini)
predicted_sigmoid = classifier_sigmoid.predict(x_test)

print(f"Test Accuracy: {classifier_sigmoid.score(x_test, y_test):.4f}")
print("\nClassification Report:")
print(metrics.classification_report(y_test, predicted_sigmoid, target_names=class_names))

cm_sigmoid = metrics.confusion_matrix(y_test, predicted_sigmoid, normalize='true')
fig, ax = plt.subplots(figsize=(10, 8))
metrics.ConfusionMatrixDisplay(cm_sigmoid, display_labels=class_names).plot(ax=ax, xticks_rotation=45)
plt.title('Confusion Matrix: Sigmoid Kernel')
plt.show()

# RBF Kernel

In [ ]:


rbf_results = []
for C in [0.1, 1, 10]:
    for gamma in ['scale', 0.01]:
        clf = svm.SVC(kernel='rbf', C=C, gamma=gamma)
        start = time.time()
        clf.fit(x_train_mini, y_train_mini)
        rbf_results.append({
            'C': C, 'gamma': gamma, 'train_acc': clf.score(x_train_mini, y_train_mini),
            'test_acc': clf.score(x_test, y_test), 'time': time.time() - start
        })

best_rbf = get_best(rbf_results)
print(f"Best RBF: {best_rbf}")

In [ ]:
# Get best RBF config
best_rbf = get_best(rbf_results)
print(f"\nBest RBF: {best_rbf}")

# Final Evaluation for RBF
classifier_rbf = svm.SVC(kernel='rbf', C=best_rbf['C'], gamma=best_rbf['gamma'])

# Train and Predict
classifier_rbf.fit(x_train_mini, y_train_mini)
predicted_rbf = classifier_rbf.predict(x_test)

# Metrics
print(f"Test Accuracy: {classifier_rbf.score(x_test, y_test):.4f}")
print("\nClassification Report:")
print(metrics.classification_report(y_test, predicted_rbf, target_names=class_names))

# Confusion Matrix
cm = metrics.confusion_matrix(y_test, predicted_rbf, normalize='true')
fig, ax = plt.subplots(figsize=(10, 8))
metrics.ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, xticks_rotation=45)
plt.title('Confusion Matrix: RBF Kernel')
plt.show()

In [ ]:
best_linear = get_best(linear_results)
best_rbf = get_best(rbf_results)
best_poly = get_best(poly_results)
best_sigmoid = get_best(sigmoid_results)

print("--- Final Best Configurations ---")
print(f"Linear:  {best_linear}")
print(f"RBF:     {best_rbf}")
print(f"Poly:    {best_poly}")
print(f"Sigmoid: {best_sigmoid}")

# Sigmoid Kernel

In [ ]:


sigmoid_results = []
for C in [0.1, 1, 10]:
    clf = svm.SVC(kernel='sigmoid', C=C, coef0=0.5)
    start = time.time()
    clf.fit(x_train_mini, y_train_mini)
    sigmoid_results.append({
        'C': C, 'coef0': 0.5, 'train_acc': clf.score(x_train_mini, y_train_mini),
        'test_acc': clf.score(x_test, y_test), 'time': time.time() - start
    })

best_sigmoid = get_best(sigmoid_results)
print(f"Best Sigmoid: {best_sigmoid}")

In [ ]:
# Get best Sigmoid config
best_sigmoid = get_best(sigmoid_results)
print(f"\nBest Sigmoid: {best_sigmoid}")

# Final Evaluation for Sigmoid
classifier_sigmoid = svm.SVC(kernel='sigmoid', C=best_sigmoid['C'], gamma='scale', coef0=best_sigmoid['coef0'])
classifier_sigmoid.fit(x_train_mini, y_train_mini)
predicted_sigmoid = classifier_sigmoid.predict(x_test)

# Metrics
print(f"Test Accuracy: {classifier_sigmoid.score(x_test, y_test):.4f}")
print("\nClassification Report:")
print(metrics.classification_report(y_test, predicted_sigmoid, target_names=class_names))

# Confusion Matrix
cm_sig = metrics.confusion_matrix(y_test, predicted_sigmoid, normalize='true')
fig, ax = plt.subplots(figsize=(10, 8))
metrics.ConfusionMatrixDisplay(cm_sig, display_labels=class_names).plot(ax=ax, xticks_rotation=45)
plt.title('Confusion Matrix: Sigmoid Kernel')
plt.show()

# Final Comparison


In [ ]:

summary = pd.DataFrame([
    {'Kernel': 'Linear', **best_linear},
    {'Kernel': 'Polynomial', **best_poly},
    {'Kernel': 'RBF', **best_rbf},
    {'Kernel': 'Sigmoid', **best_sigmoid}
])
display(summary.sort_values('test_acc', ascending=False))